In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q sentence-transformers

import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer

pairs = pd.read_csv("/kaggle/input/datasets/kylin0338/aspectfinal/aspect_sentiment_final.csv")
print(pairs.shape)
print(pairs.columns.tolist())

(43230, 10)
['review_id', 'asin', 'rating', 'sentence', 'aspect', 'p_neg', 'p_neu', 'p_pos', 'aspect_sentiment', 'sentiment_final']


In [2]:
pairs["asp_sent"] = pairs["aspect"] + " (" + pairs["sentiment_final"] + ")"

idx = (pairs.groupby(["review_id", "asin", "sentence"], as_index=False)
            .agg(rating=("rating", "first"),
                 aspects=("asp_sent", lambda s: ", ".join(sorted(set(s)))),
                 aspect_list=("aspect", lambda s: "|".join(sorted(set(s)))),
                 sentiments=("sentiment_final", lambda s: "|".join(sorted(set(s))))))

idx = idx.reset_index(drop=True)
idx["sentence"] = idx["sentence"].str.slice(0, 400)
print(idx.shape)
idx.head()

(37051, 7)


,review_id,asin,sentence,rating,aspects,aspect_list,sentiments
0,0,B00EP2BN00,"As well, I should have listened to the comment...",2,screen (negative),screen,negative
1,0,B00EP2BN00,On about the 32nd day the front screen mysteri...,2,screen (negative),screen,negative
2,0,B00EP2BN00,The speaker has deteriorated as well...I'm sor...,2,sound (negative),sound,negative
3,0,B00EP2BN00,now the battery is draining every 24 to 48 hrs.,2,battery (negative),battery,negative
4,1,B01FJT7E3M,This phone 📱 is so good picture r,5,camera (positive),camera,positive


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

emb = model.encode(idx["sentence"].tolist(), batch_size=256,
                   normalize_embeddings=True, show_progress_bar=True)
print(emb.shape)          # (number of sentences, 384)

np.save("/kaggle/working/search_emb.npy", emb.astype("float16"))
idx.to_csv("/kaggle/working/search_index.csv", index=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

(37051, 384)


In [6]:
def search(query, asin=None, aspect=None, sentiment=None, k=8, min_sim=0.30):
    q = model.encode([query], normalize_embeddings=True)[0]
    sims = emb @ q
    mask = np.ones(len(idx), bool)
    if asin:
        mask &= (idx["asin"] == asin).values
    if aspect:
        mask &= idx["aspect_list"].str.contains(rf"(?:^|\|){aspect}(?:\||$)").values
    if sentiment:
        mask &= idx["sentiments"].str.contains(sentiment).values
    sims = np.where(mask, sims, -1)
    order = np.argsort(-sims)[: k * 4]
    cand = [i for i in order if sims[i] >= min_sim]
    out = idx.loc[cand].copy()
    out["similarity"] = sims[cand]
    out["_norm"] = out["sentence"].str.lower().str.replace(r"[^a-z0-9 ]", "", regex=True).str.strip()
    out = out.drop_duplicates("_norm").head(k).drop(columns="_norm")
    return out
    

pd.set_option("display.max_colwidth", 140)
for q in ["does it overheat", "screen cracks easily", "good for travel", "customer support was unhelpful"]:
    print("\n###", q)
    print(search(q)[["similarity", "sentence", "aspects"]].to_string(index=False))


### does it overheat
 similarity                                                                                                                                                                          sentence                             aspects
   0.732944                                                                                                                                   No overheating and the speaker has great sound.                    sound (positive)
   0.652833 The first day I used it, It overheated A LOT, I was worried because I didn't want to return it after all the days I waited and the complication of waiting again for another one.         customer_service (negative)
   0.634423                                                                                          It overheats, and the screen is not very good quality because I've cracked both of mine. build (negative), screen (negative)
   0.597011                                                 Due to its sle

In [7]:
import os
for f in ["search_emb.npy", "search_index.csv"]:
    print(f, round(os.path.getsize(f"/kaggle/working/{f}") / 1e6, 1), "MB")

search_emb.npy 28.5 MB
search_index.csv 5.6 MB
